# Classification - Syeda

**Classification**\
Classify stress levels into low, moderate, high\
Dataset to be used: df2, but would need dropping of gender --> df_class


In [ ]:
import numpy as np
from sklearn.model_selection import LeaveOneOut, KFold, StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.dummy import DummyClassifier
from tqdm import tqdm       #for progress bar

In [ ]:
#load data
df2.head()
df_class = df2.drop(columns = 'Gender')
df_class.head()
# sns.pairplot(df_class, size=3)

#split into features and labels
X = df_class.drop(columns='Stress_Level')        #features
y = df_class['Stress_Level']                     #labels

assert X.shape == (2000,6)
assert y.shape == (2000,)

**Baseline**\
A moodel that ignores input and uses majority voting to choose label.\
2-level CV fold.\
Split data into training and test sets.\

In [ ]:
#initialise leave-one-out cross validation, save it as CV_loo
CV_loo = LeaveOneOut()  

accuracies_bl = []

#loop through the folds
for fold, (train_index, test_index) in tqdm(enumerate(CV_loo.split(X)), 
                                            desc="Cross validation fold (baseline)", 
                                            total=CV_loo.get_n_splits(X)):
    #data split
    X_train, y_train = X.iloc[train_index,:], y[train_index]    #iloc: integer-location based indexing 
    X_test, y_test = X.iloc[test_index, :], y[test_index]

    #apply model 
    model = DummyClassifier(strategy = 'most_frequent')
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    accuracies_bl.append(acc)

#results for baseline model  
#results for multinomail logistic regression 
print(f"Mean accuracy for baseline model: {np.mean(accuracies_bl):.4f}")
print(f"Mean error for baseline model: {1-np.mean(accuracies_bl):.4f}")

Cross validation fold (baseline): 100%|██████████| 2000/2000 [00:01<00:00, 1522.47it/s]

Mean accuracy for baseline model: 0.5145
Mean error for baseline model: 0.4855


**Multinomial logistic regression**\
Regularization parameter λ≥, using softmax activation for our multi-class problem.\
(Assumptions for logistic regression: independent observations, binary dependet variables, linear relationship
between independent variables and log odds, no outliers, large sample size)

In [ ]:
#initialise leave-one-out cross validation, save it as CV_loo
CV_loo_lr = LeaveOneOut()  

accuracies_lr = []

#loop through the folds
for fold, (train_index, test_index) in tqdm(enumerate(CV_loo_lr.split(X)), 
                                            desc="Cross validation fold (multinormial logistic regression)", 
                                            total=CV_loo_lr.get_n_splits(X)):
    #data split
    X_train, y_train = X.iloc[train_index,:], y[train_index]    #iloc: integer-location based indexing 
    X_test, y_test = X.iloc[test_index, :], y[test_index]


    #apply model 
    model = LogisticRegression(
                                # multi_class = 'multinomial',   #deprecated variable, default is multinomialf or n>2
                               max_iter = 1000,
                               solver = 'lbfgs', 
                               C = 1.0)     #C is inverse of regularization strength (lambda), default is 1
    
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    accuracies_lr.append(acc)

#results for multinomail logistic regression 
print(f"Mean accuracy for logistic regression: {np.mean(accuracies_lr):.4f}")
print(f"Mean error for logistic regression: {1-np.mean(accuracies_lr):.4f}")

Cross validation fold (multinormial logistic regression): 100%|██████████| 2000/2000 [00:36<00:00, 54.69it/s]

Mean accuracy for logistic regression: 0.8275
Mean error for logistic regression: 0.1725


**Method 2: KNN**\
Complexity controlling parameter k= 1,2...\
Evaluate performance of models with varying k-values using 2-fold leave-one-out cross-validation.\
Split data into training and test sets for each fold.\

In [ ]:
#maximum number of neighbors
K_neighbors = [1, 3, 7, 9, 11, 15,31]

#initialise leave-one-out cross validation, save it as CV_loo
CV_loo_knn = LeaveOneOut()  

#accuracies for each k
accuracies_knn = {k: [] for k in K_neighbors}

#loop through the folds
for fold, (train_index, test_index) in tqdm(enumerate(CV_loo_knn.split(X)), 
                                            desc="Cross validation fold (KNN)", 
                                            total=CV_loo_knn.get_n_splits(X)):
    #data split
    X_train, y_train = X.iloc[train_index,:], y[train_index]    #iloc: integer-location based indexing 
    X_test, y_test = X.iloc[test_index, :], y[test_index]

    for k in K_neighbors:
        
        #apply model 
        model = KNeighborsClassifier(n_neighbors = k)
        model.fit(X_train, y_train)
        
        y_pred = model.predict(X_test)
        
        acc = accuracy_score(y_test, y_pred)
        
        accuracies_knn[k].append(acc)

#results for KNN for each k  
for k in K_neighbors:
    print(f"KNN with k={k}: mean accuracy: {np.mean(accuracies_knn[k]):.4f}, mean error: {1-np.mean(accuracies_knn[k]):.4f}")

Cross validation fold (KNN): 100%|██████████| 2000/2000 [00:21<00:00, 93.05it/s]

KNN with k=1: mean accuracy: 0.8915, mean error: 0.1085
KNN with k=3: mean accuracy: 0.9020, mean error: 0.0980
KNN with k=7: mean accuracy: 0.9145, mean error: 0.0855
KNN with k=9: mean accuracy: 0.9180, mean error: 0.0820
KNN with k=11: mean accuracy: 0.9115, mean error: 0.0885
KNN with k=15: mean accuracy: 0.9090, mean error: 0.0910
KNN with k=31: mean accuracy: 0.9115, mean error: 0.0885


**Comparing the three models: Baseline, Multinomal Logistic Regression, KNN**\
2-fold cross-validation will be used, with outer 10-fold stratified CV and inner 5-fold stratified CV.\
In the 5 inner folds, the parameters will be tuned (k or KNN, $\lambda$ for logistic).\


In [ ]:
from sklearn.metrics import accuracy_score

def two_level_cv_classification(X, y, outer_folds=10, inner_folds=5):
    
    #initialise outer stratified k-fold CV with 10 folds, inner with 5
    CV_outer = StratifiedKFold(outer_folds, shuffle = True)
    CV_inner = StratifiedKFold(inner_folds, shuffle = True)
    
    #for McNemar's test later:
    all_preds = {
        "logreg": np.zeros(len(y)),
        "knn": np.zeros(len(y)),
        "baseline": np.zeros(len(y)),
        "true": np.zeros(len(y)),
    }
    
    #complexity parameters for logistic, KNN respectively
    lambda_values = [0.01, 0.1, 1, 10, 100]
    k_values = [1, 3, 7, 9, 11, 15, 31]
    
    #to store results in each outer fold
    results1 = []
    
    #outer loop
    for i, (outer_train_idx, outer_test_idx) in enumerate(CV_outer.split(X, y)):       
        
        #split data into train and test sets
        X_train_outer, X_test_outer = X.iloc[outer_train_idx], X.iloc[outer_test_idx]
        y_train_outer, y_test_outer = y.iloc[outer_train_idx], y.iloc[outer_test_idx]
    
        #-----------multinomial logreg inner CV----------------
        avg_errors_lr = []
        for lam in lambda_values:
            inner_errors_lr = []
            for inner_train_idx, inner_test_idx in CV_inner.split(X_train_outer, y_train_outer):
                X_train_inner, X_test_inner = X_train_outer.iloc[inner_train_idx], X_train_outer.iloc[inner_test_idx]  
                y_train_inner, y_test_inner = y_train_outer.iloc[inner_train_idx], y_train_outer.iloc[inner_test_idx]  
                
                model = LogisticRegression(max_iter = 1000,solver = 'lbfgs', C = 1/lam)
                model.fit(X_train_inner, y_train_inner)
                
                y_pred_inner = model.predict(X_test_inner)
                inner_errors_lr.append(1 - accuracy_score(y_test_inner, y_pred_inner))
            avg_errors_lr.append(np.mean(inner_errors_lr))
        best_lambda = lambda_values[np.argmin(avg_errors_lr)]
        
        #-----------KNN inner CV----------------
        avg_errors_knn = []
        for k in k_values:
            inner_errors_knn = []
            for inner_train_idx, inner_test_idx in CV_inner.split(X_train_outer, y_train_outer):
                X_train_inner, X_test_inner = X_train_outer.iloc[inner_train_idx], X_train_outer.iloc[inner_test_idx]  
                y_train_inner, y_test_inner = y_train_outer.iloc[inner_train_idx], y_train_outer.iloc[inner_test_idx]  
                
                model = KNeighborsClassifier(n_neighbors = k)
                model.fit(X_train_inner, y_train_inner)
                
                y_pred_inner = model.predict(X_test_inner)
                inner_errors_knn.append(1 - accuracy_score(y_test_inner, y_pred_inner))
            avg_errors_knn.append(np.mean(inner_errors_knn))
        best_k = k_values[np.argmin(avg_errors_knn)]
        
        
        #-------baseline model has nothing to loop over since no complexity parameter-----
        
        #statistical evaluation of outer splits
        
        #logreg
        model_lr = LogisticRegression(max_iter = 1000,solver = 'lbfgs', C = 1/best_lambda)      
        model_lr.fit(X_train_outer, y_train_outer)
        y_pred_lr = model_lr.predict(X_test_outer)
        E_lr = 1 - accuracy_score(y_test_outer, y_pred_lr)
        
        #knn
        model_knn = KNeighborsClassifier(n_neighbors = best_k)                                  
        model_knn.fit(X_train_outer, y_train_outer)
        y_pred_knn = model_knn.predict(X_test_outer)
        E_knn = 1 - accuracy_score(y_test_outer, y_pred_knn)
        
        #baseline
        model_bl = DummyClassifier(strategy = 'most_frequent')                                  
        model_bl.fit(X_train_outer, y_train_outer)
        y_pred_bl = model_bl.predict(X_test_outer)
        E_bl = 1 - accuracy_score(y_test_outer, y_pred_bl)
        
        #store predictions for outer test indices (for McNemar's test)
        all_preds['logreg'][outer_test_idx] = y_pred_lr
        all_preds['knn'][outer_test_idx] = y_pred_knn
        all_preds['baseline'][outer_test_idx] = y_pred_bl
        all_preds['true'][outer_test_idx] = y_test_outer
        
        #store results 
        results1.append({
            "Outer fold (i)": i + 1,
            "h*_i (KNN)": best_k,
            "E_test (KNN)": round(E_knn, 3),
            "λ*_i (LogReg)": best_lambda,
            "E_test (LogReg)": round(E_lr, 3),
            "E_test (Baseline)": round(E_bl, 3)
        })

    return pd.DataFrame(results1), all_preds

In [ ]:
results_df, all_preds = two_level_cv_classification(X, y)
print(results_df)
print("\n")
print(all_preds)

   Outer fold (i)  h*_i (KNN)  E_test (KNN)  λ*_i (LogReg)  E_test (LogReg)  \
0               1          15         0.090           1.00            0.165   
1               2          11         0.090          10.00            0.185   
2               3           7         0.065          10.00            0.180   
3               4           3         0.105           0.01            0.195   
4               5          11         0.115           0.10            0.190   
5               6          15         0.070           0.10            0.200   
6               7           7         0.105           0.01            0.170   
7               8          15         0.075           1.00            0.145   
8               9          11         0.095           1.00            0.185   
9              10           9         0.100           0.01            0.155   

   E_test (Baseline)  
0              0.490  
1              0.485  
2              0.485  
3              0.485  
4              

**Statistical Evalaution of the three models**\
Using Setup I: McNemar's test: check whether models behave differently or not, on the same test samples.\
Use the all_preds variable in CV function to get statistical results.\
p-values and confidence intervals for three pairwise tests.\
Baseline vs. KNN, Baseline vs. Logreg, KNN vs Logreg.

In [ ]:
from statsmodels.stats.contingency_tables import mcnemar

#get labels from each model, as well as true labels
y_true = np.array(all_preds["true"])
y_lr = np.array(all_preds["logreg"])
y_knn = np.array(all_preds["knn"])
y_bl = np.array(all_preds["baseline"])

def mcnemar_test(y_true, pred1, pred2, model1, model2):
    
    #compare whether results from both models equal or not
    a_correct = pred1 == y_true
    b_correct = pred2 == y_true
    
    #values for contingency table
    n01 = np.sum(~a_correct & b_correct)        #no. of cases where model1 wrong, model2 right
    n10 = np.sum(a_correct & ~b_correct)        #no. of cases where model1 right, model2 wrong
    
    #contingency table, carry out mcnemar test
    table = [[0, n01],[n10, 0]]
    mcnemar_result = mcnemar(table, exact=False, correction=True)
    
    #confidence intervals to see difference in accuracies
    n_total = n01 + n10
    if n_total > 0:
        diff = (n10 - n01)/n_total
        se = np.sqrt(1/n_total)               #standard error
        ci_low = diff - (1.96 * se)
        ci_high = diff + (1.96 * se)
    else:
        diff, ci_low, ci_high = 0, 0, 0
    
    print(f"{model1} vs {model2}:")
    print(f"diff = {diff:.4f}, p-value = {mcnemar_result.pvalue:.3e}, 95% CI: [{ci_low:.4f},{ci_high:.4f}]")
    
#pairwise comparisons
mcnemar_test(y_true, y_lr, y_knn, "Logreg", "KNN")
mcnemar_test(y_true, y_bl, y_lr, "Baseline", "Logreg")
mcnemar_test(y_true, y_bl, y_knn, "Baseline", "KNN")

Logreg vs KNN:
diff = -0.4433, p-value = 3.916e-18, 95% CI: [-0.5428,-0.3438]
Baseline vs Logreg:
diff = -0.6894, p-value = 3.332e-94, 95% CI: [-0.7549,-0.6239]
Baseline vs KNN:
diff = -0.9293, p-value = 4.482e-161, 95% CI: [-0.9966,-0.8621]


**Train logistic regression model using suitable $\lambda$ = 0.01**\
This value was chosen since it appeared most often in the 2-level CV.\
Check which features are more relevant by finding their coefficients; the higher the coefficient, the higher that feature was weighted.

In [ ]:
suitable_lambda = 0.01
final_lr_model = LogisticRegression(max_iter = 1000,solver = 'lbfgs', C = 1/suitable_lambda)
final_lr_model.fit(X, y)

#find coefficients of each feature
coefficients = pd.DataFrame(
    final_lr_model.coef_,
    columns = X.columns, 
    index = [f"Class {stress_level_class}" for stress_level_class in  final_lr_model.classes_]
)

print(coefficients)

         Study_Hours_Per_Day  Extracurricular_Hours_Per_Day  \
Class 0            -4.649550                       1.000170   
Class 1             1.529938                      -0.352726   
Class 2             3.119612                      -0.647445   

         Sleep_Hours_Per_Day  Social_Hours_Per_Day  \
Class 0             2.092662              0.877780   
Class 1            -0.347067             -0.288810   
Class 2            -1.745595             -0.588969   

         Physical_Activity_Hours_Per_Day    Grades  
Class 0                         0.896792  0.331666  
Class 1                        -0.324818 -0.245299  
Class 2                        -0.571974 -0.086367  
